# Multi-dataset immune integration: Data Loading & QC

Load 7 multiome datasets (RNA + ATAC), harmonize annotations and metadata, compute QC metrics, and save combined h5ad objects. **No cell or gene filtering** — filtering is done in the training notebook after inspecting both RNA and ATAC QC metrics.

**Datasets:**
1. Bone marrow (NeurIPS 2021) — 13 batches, ~69k cells
2. TEA-seq PBMC (GSE158013) — 7 samples, ~52k cells
3. NEAT-seq CD4 T (GSE178707) — 2 lanes, ~8.5k cells
4. Crohn's PBMC (GSE244831) — 13 samples, ~76k cells
5. COVID infant PBMC (GSE239799) — 43 samples, no annotations
6. Lung/Spleen immune (GSE319044) — 16 samples, ~54k cells
7. Infant/Adult Spleen (GSE311423) — 5 samples, no annotations

**Outputs:**
- `results/immune_integration/adata_rna.h5ad` — all cells, with RNA QC metrics + doublet scores
- `results/immune_integration/adata_atac_tiles.h5ad` — all cells, with ATAC QC metrics

See `PLAN.md` for full details.

In [ ]:
import gc
import os

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib import rcParams

from data_loading_utils import (
    load_bone_marrow,
    load_covid_pbmc_gse239799,
    load_crohns_pbmc_gse244831,
    load_infant_adult_spleen_gse311423,
    load_lung_spleen_gse319044,
    load_neat_seq_cd4t,
    load_tea_seq_pbmc,
)

rcParams["pdf.fonttype"] = 42

In [ ]:
results_folder = "results/immune_integration/"

In [ ]:
os.makedirs(results_folder, exist_ok=True)

## 1. Load all datasets (RNA)

In [ ]:
adata_bm = load_bone_marrow()

In [ ]:
adata_tea = load_tea_seq_pbmc()

In [ ]:
adata_neat = load_neat_seq_cd4t()

In [ ]:
adata_crohns = load_crohns_pbmc_gse244831()

In [ ]:
adata_covid = load_covid_pbmc_gse239799()

In [ ]:
adata_lung_spleen = load_lung_spleen_gse319044()

In [ ]:
adata_spleen311 = load_infant_adult_spleen_gse311423()

### Per-dataset summary

In [ ]:
datasets = {
    "bone_marrow": adata_bm,
    "pbmc_tea_seq": adata_tea,
    "neat_seq_cd4t": adata_neat,
    "crohns_pbmc": adata_crohns,
    "covid_pbmc": adata_covid,
    "lung_spleen_gse319044": adata_lung_spleen,
    "infant_adult_spleen": adata_spleen311,
}

summary_rows = []
for name, ad in datasets.items():
    n_annot = ad.obs["harmonized_annotation"].notna().sum()
    summary_rows.append(
        {
            "dataset": name,
            "n_cells": ad.n_obs,
            "n_genes": ad.n_vars,
            "n_batches": ad.obs["batch"].nunique(),
            "n_donors": ad.obs["donor"].nunique(),
            "n_annotated": n_annot,
            "pct_annotated": f"{100 * n_annot / ad.n_obs:.1f}%",
        }
    )
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## 2. Combine RNA

In [ ]:
# Gene intersection across all datasets (ENSEMBL IDs)
gene_sets = [set(ad.var_names) for ad in datasets.values()]
common_genes = gene_sets[0]
for gs in gene_sets[1:]:
    common_genes = common_genes.intersection(gs)
common_genes = sorted(common_genes)
print(f"Common genes across all datasets: {len(common_genes)}")

# Subset each dataset to common genes
adatas_subset = [ad[:, common_genes].copy() for ad in datasets.values()]

# Free individual datasets
del adata_bm, adata_tea, adata_neat, adata_crohns, adata_covid, adata_lung_spleen, adata_spleen311
del datasets
gc.collect()

In [ ]:
# Concatenate
adata = anndata.concat(adatas_subset, join="inner")
adata.obs_names_make_unique()
print(f"Combined: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Datasets: {adata.obs['dataset'].value_counts().to_dict()}")

# Ensure uint16 counts layer
adata.X = adata.X.tocsr().astype(np.uint16)
adata.layers["counts"] = adata.X.copy()
print(f"X dtype: {adata.X.dtype}")

del adatas_subset
gc.collect()

## 3. QC — RNA metrics

In [ ]:
# Identify mitochondrial genes
adata.var["mt"] = adata.var["SYMBOL"].str.startswith("MT-") | adata.var["SYMBOL"].str.startswith("mt-")
print(f"MT genes: {adata.var['mt'].sum()}")

# Compute QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, layer="counts")
adata.obs.rename(
    columns={
        "total_counts": "n_counts_rna",
        "n_genes_by_counts": "n_genes_rna",
        "pct_counts_mt": "mt_frac",
    },
    inplace=True,
)
# mt_frac is percentage, convert to fraction
adata.obs["mt_frac"] = adata.obs["mt_frac"] / 100.0

In [ ]:
# QC distributions per dataset
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for metric, ax, title in zip(
    ["n_counts_rna", "n_genes_rna", "mt_frac"],
    axes,
    ["Total RNA counts", "N genes detected", "MT fraction"],
    strict=True,
):
    for ds in adata.obs["dataset"].unique():
        vals = adata.obs.loc[adata.obs["dataset"] == ds, metric]
        ax.hist(vals, bins=100, alpha=0.5, label=ds, density=True)
    ax.set_title(title)
    ax.set_xlabel(metric)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Scrublet doublet detection per batch
sc.pp.scrublet(adata, batch_key="batch")
adata.obs.rename(columns={"doublet_score": "doublet_score"}, inplace=True)
print(
    f"Predicted doublets: {adata.obs['predicted_doublet'].sum()} / {adata.n_obs} ({100 * adata.obs['predicted_doublet'].mean():.1f}%)"
)

In [ ]:
# Doublet score distribution
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(adata.obs["doublet_score"], bins=100)
ax.set_xlabel("Doublet score")
ax.set_ylabel("Frequency")
ax.set_title("Scrublet doublet scores")
plt.tight_layout()
plt.show()

## 4. Save RNA (all cells, before filtering)

Cell filtering will be done in the training notebook after inspecting both RNA and ATAC QC metrics.

In [ ]:
# Save RNA (unfiltered — all cells with QC metrics + doublet scores)
rna_path = os.path.join(results_folder, "adata_rna.h5ad")
adata.write_h5ad(rna_path)
print(f"Saved RNA: {rna_path}")
print(f"  Shape: {adata.shape}")
print(f"  obs columns: {list(adata.obs.columns)}")

# Save obs as CSV for inspection
obs_path = os.path.join(results_folder, "obs_metadata.csv")
adata.obs.to_csv(obs_path)
print(f"Saved obs metadata: {obs_path}")

# RNA summary
print("\nDatasets:")
for ds in sorted(adata.obs["dataset"].unique()):
    n = (adata.obs["dataset"] == ds).sum()
    n_annot = adata.obs.loc[adata.obs["dataset"] == ds, "harmonized_annotation"].notna().sum()
    print(f"  {ds}: {n} cells ({n_annot} annotated)")
print(f"\nTotal batches: {adata.obs['batch'].nunique()}")
print(f"Total donors: {adata.obs['donor'].nunique()}")
print(f"Total cell types (harmonized): {adata.obs['harmonized_annotation'].dropna().nunique()}")

## 5. ATAC tile loading

Load ATAC fragments into 1000bp tile matrix using SnapATAC2 via cell2state.

In [ ]:
from cell2state.utils.load_atac_snapatac2 import concatenate_h5ad

genome_ref = "/nfs/srpipe_references/downloaded_from_10X/refdata-cellranger-arc-GRCh38-2020-A-2.0.0/"

adata_atac = concatenate_h5ad(
    adata,
    genome_ref=genome_ref,
    bin_size=1000,
    counting_strategy="insertion",
    max_frag_size_split=120,
)

# Cast to uint16
adata_atac.X = adata_atac.X.tocsr().astype(np.uint16)
adata_atac.layers["counts"] = adata_atac.X.copy()

print(f"ATAC tiles: {adata_atac.n_obs} cells x {adata_atac.n_vars} tiles")
print(f"X dtype: {adata_atac.X.dtype}")

## 6. ATAC QC metrics

In [ ]:
# Compute ATAC QC
sc.pp.calculate_qc_metrics(adata_atac, inplace=True)
adata_atac.obs.rename(
    columns={
        "total_counts": "n_counts_atac",
        "n_genes_by_counts": "n_features_atac",
    },
    inplace=True,
)

print(f"ATAC cells: {adata_atac.n_obs}")
print(f"Median ATAC counts: {adata_atac.obs['n_counts_atac'].median():.0f}")
print(f"Median ATAC features: {adata_atac.obs['n_features_atac'].median():.0f}")

In [ ]:
# ATAC QC distributions per dataset
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for metric, ax, title in zip(
    ["n_counts_atac", "n_features_atac"],
    axes,
    ["Total ATAC counts", "N ATAC features"],
    strict=True,
):
    for ds in adata_atac.obs["dataset"].unique():
        vals = adata_atac.obs.loc[adata_atac.obs["dataset"] == ds, metric]
        ax.hist(vals, bins=100, alpha=0.5, label=ds, density=True)
    ax.set_title(title)
    ax.set_xlabel(metric)
axes[-1].legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 7. Save ATAC

In [ ]:
# Save ATAC
atac_path = os.path.join(results_folder, "adata_atac_tiles.h5ad")
adata_atac.write_h5ad(atac_path)
print(f"Saved ATAC: {atac_path}")
print(f"  Shape: {adata_atac.shape}")

In [ ]:
# Final summary
print("=== Final Summary ===")
print(f"RNA: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"ATAC: {adata_atac.n_obs} cells x {adata_atac.n_vars} tiles")
print("\nDatasets:")
for ds in sorted(adata.obs["dataset"].unique()):
    n_rna = (adata.obs["dataset"] == ds).sum()
    n_atac = (adata_atac.obs["dataset"] == ds).sum() if "dataset" in adata_atac.obs.columns else 0
    n_annot = adata.obs.loc[adata.obs["dataset"] == ds, "harmonized_annotation"].notna().sum()
    print(f"  {ds}: {n_rna} RNA cells, {n_atac} ATAC cells ({n_annot} annotated)")
print(f"\nTotal batches: {adata.obs['batch'].nunique()}")
print(f"Total donors: {adata.obs['donor'].nunique()}")
print(f"Total cell types (harmonized): {adata.obs['harmonized_annotation'].dropna().nunique()}")